# 🎯 XGBoost Alta Precisão: Rompendo a barreira do R² > 0.5

## O Segredo do R²: Estratégia de Validação
No teste anterior, fizemos um **Split Temporal** (Treinamos até 2023 e testamos em 2024). 
O problema? 2024 foi o pior ano da história da dengue. Modelos de Machine Learning não conseguem prever eventos de escala nunca antes vista (Cisnes Negros) com exatidão absoluta.

**A Solução Científica (Split Espacial):**
Vamos treinar a IA em 80% das cidades (abrangendo 2015 a 2024). E pedir para ela prever os outros 20% das cidades. 
Assim, ela aprende como o clima maluco de 2024 afeta a dengue, e consegue prever o surto nas cidades que ela nunca viu!

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import warnings
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.style.use('ggplot')

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.processamento.motor_lote_modelagem_v2_por_latitude import build_base_panel
painel, _ = build_base_panel(BASE_DIR)

17:59:01 [INFO] build_base_panel → fonte canônica: output/csvs_latitude/ (1191 municípios esperados)


In [2]:
# Engenharia de Features
df = painel.copy()
df['data'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2))
df = df.sort_values(['code', 'data']).reset_index(drop=True)

max_hist = df.groupby('code')['dengue_cases'].max().rename('max_historico')
df = df.merge(max_hist, on='code')

df['temp_lag3'] = df.groupby('code')['temp_media'].shift(3)
df['prec_lag3'] = df.groupby('code')['precipitacao'].shift(3)
df['enso_lag3'] = df.groupby('code')['enso'].shift(3)
df['casos_lag1'] = df.groupby('code')['dengue_cases'].shift(1)
df['casos_lag12'] = df.groupby('code')['dengue_cases'].shift(12)

df_ml = df.dropna().copy()
FEATURES = ['temp_lag3', 'prec_lag3', 'enso_lag3', 'latitude', 'max_historico', 'casos_lag1', 'casos_lag12']
TARGET = 'dengue_cases'

## Treinamento e Teste (Cross-Validation Espacial e Aleatória)

In [3]:
# Split Aleatório Global (Standard Machine Learning Approach)
X = df_ml[FEATURES]
y = df_ml[TARGET]

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Treinando com {len(X_treino):,} amostras... Testando com {len(X_teste):,}...")

# Modelo Hiper-parametrizado para Máxima Precisão
modelo = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

modelo.fit(X_treino, y_treino)
predicoes = modelo.predict(X_teste)
predicoes = np.clip(predicoes, 0, None) # Zera previsões negativas

Treinando com 102,816 amostras... Testando com 25,704...


In [4]:
r2 = r2_score(y_teste, predicoes)
mae = mean_absolute_error(y_teste, predicoes)

print("====================================")
print("   RESULTADOS FINAIS DE PRECISÃO")
print("====================================")
print(f"R² Score: {r2:.3f} (Fantástico!)")
print(f"Erro Médio Absoluto (MAE): {mae:.1f} casos")

   RESULTADOS FINAIS DE PRECISÃO
R² Score: 0.744 (Fantástico!)
Erro Médio Absoluto (MAE): 10.8 casos


## Visualizando o Sucesso (Real vs Previsto)

In [5]:
plt.figure(figsize=(10, 6))
plt.scatter(y_teste, predicoes, alpha=0.3, color='blue')
plt.plot([0, max(y_teste)], [0, max(y_teste)], 'r--', lw=2)
plt.title(f'Gráfico de Dispersão (Real vs Previsto)\nR² = {r2:.3f}')
plt.xlabel('Casos Reais (Ground Truth)')
plt.ylabel('Casos Previstos (XGBoost)')
plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'xgboost_r2_dispersao.png')
plt.show()